# Stage 04 · Prospective forecast, quality audit, berth gate, and ETA benchmarks

Prompt 3 prospective case generation is retained. Prompt 4 replaces probability-like confidence with an independent-component data-quality index; Prompt 6 audits the deterministic berth-slot gate; Prompt 7 adds fixed and online-corrected ETA benchmarks with prospective intervals.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys
import pandas as pd

root = Path(os.environ.get("MFAR_CODE_ROOT", Path.cwd()))
if not (root / "src" / "mfar_paths.py").is_file():
    root = Path.cwd().parent
if not (root / "src" / "mfar_paths.py").is_file():
    raise FileNotFoundError("Run the notebook from the repository or set MFAR_CODE_ROOT")
sys.path.insert(0, str(root.resolve()))

from src.mfar_paths import *
from src.mfar_prompt4_7 import run_stage4
STARTED_AT = datetime.now(timezone.utc)


In [ ]:
NOTEBOOK_NAME = "04_No_Intervention_Forecast.ipynb"
validate_writable_directory(STAGE_DIRS[4], NOTEBOOK_NAME, 4)


In [ ]:
state = validate_csv_input(STAGE_03_DIR / "03_input_state_enhanced.csv",
    ["grid_time", "mmsi", "operational_status", "origin", "destination",
     "is_at_berth", "berth_episode_id", "elapsed_berth_min",
     "predicted_berth_release_time", "operational_phase"], NOTEBOOK_NAME, 4)
for col in ["grid_time", "predicted_berth_release_time"]:
    state[col] = pd.to_datetime(state[col], errors="coerce")
episodes = validate_csv_input(STAGE_03_DIR / "03_service_call_history.csv",
    ["mmsi", "berth_episode_id", "berth_entry_time", "observed_end",
     "observed_release_time", "port_id", "episode_class",
     "eligible_for_turnaround_calibration"], NOTEBOOK_NAME, 4)
rates = validate_csv_input(VEHICLE_ARRIVAL_PATH,
    ["port_id", "time_start", "time_end", "car_arrival_rate_30min",
     "motorcycle_arrival_rate_30min"], NOTEBOOK_NAME, 4)
profiles = pd.read_csv(CONFIG_DIR / "vessel_profiles.csv")
berths = pd.read_csv(CONFIG_DIR / "terminal_berths.csv")
queue, event_log, forecast, summary, eta_audit, departure_audit = run_stage4(
    state, episodes, rates, profiles, berths, STAGE_04_DIR, CONFIG_DIR)
display(summary); display(eta_audit); display(departure_audit)


In [ ]:
NOTEBOOK_NAME = "04_No_Intervention_Forecast.ipynb"
stage_dir = STAGE_DIRS[4]
files = sorted(stage_dir.glob("04_*"))
write_execution_metadata(4, NOTEBOOK_NAME, STARTED_AT,
    [CONFIG_DIR / "pipeline_parameters.csv", STAGE_03_DIR / "03_service_call_history.csv"], {},
    {"prospective_cases": len(forecast),
     "matched_departures": int(forecast["observed_departure_time"].notna().sum()),
     "positive_wait_cases": int(forecast["predicted_wait_min"].gt(0).sum())}, files)
